# RSNA 2024 kaggle data challenge
## Basic descriptive statistics of the dataset

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import nibabel as nib
import os
from tqdm import tqdm

In [2]:
descriptions = pd.read_csv("data/train_series_descriptions.csv")
train = pd.read_csv("data/train.csv")
study_ids = np.unique(descriptions["study_id"].values)

In [3]:
descriptions

,study_id,series_id,series_description
0,4003253,702807833,Sagittal T2/STIR
1,4003253,1054713880,Sagittal T1
2,4003253,2448190387,Axial T2
3,4646740,3201256954,Axial T2
4,4646740,3486248476,Sagittal T1
...,...,...,...
6289,4287160193,1507070277,Sagittal T2/STIR
6290,4287160193,1820446240,Axial T2
6291,4290709089,3274612423,Sagittal T2/STIR
6292,4290709089,3390218084,Axial T2


## Number of subjects

In [4]:
nb_subjects = len(np.unique(descriptions["study_id"].values))
print("Number of subjects : ", nb_subjects)

Number of subjects :  1975


In [5]:
nb_scan = descriptions.groupby("study_id").count().values[:,0]

In [6]:
for i in range(1, 8):
    print(f"Number of subjects with {i} volumes :", np.sum(nb_scan==i))

Number of subjects with 1 volumes : 0
Number of subjects with 2 volumes : 3
Number of subjects with 3 volumes : 1632
Number of subjects with 4 volumes : 309
Number of subjects with 5 volumes : 30
Number of subjects with 6 volumes : 1
Number of subjects with 7 volumes : 0


In [7]:
sequence_types = list(np.unique(descriptions["series_description"].values))
sequence_types

['Axial T2', 'Sagittal T1', 'Sagittal T2/STIR']

In [8]:
sequence_lists = descriptions.groupby(["study_id"]).apply(lambda x: list(zip(x['series_description']))).reset_index().values[:,1]

In [9]:
# Check if there is consistancy for subjects with 3 volumes
# i.e sequences are Axial T2, Sagittal 1, Sagittal T2/STIR

check = True
for i, seq in enumerate(sequence_lists[nb_scan==3]) : 
    seq.sort()
    # print(seq)
    if seq!=[('Axial T2',), ('Sagittal T1',), ('Sagittal T2/STIR',)] : 
        print(study_ids[i])
        check = False
print(seq)

[('Axial T2',), ('Sagittal T1',), ('Sagittal T2/STIR',)]


In [10]:
# Check if every subject have Sagittal T1 volume

flag = True
count = 0
for i, study_id in enumerate(study_ids): 
    check=False
    for seq in sequence_lists[i]:
        if seq[0]=="Sagittal T1":
            check=True
            
    if not check : 
        # print(study_id)
        count+=1
        flag = False     

print("Number of subjects without Sagittal T1 scan :", count)        

Number of subjects without Sagittal T1 scan : 2


In [11]:
# Check if every subject have Axial T2 volume

flag = True
count = 0
for i, study_id in enumerate(study_ids): 
    check=False
    for seq in sequence_lists[i]:
        if seq[0]=="Axial T2":
            check=True
            
    if not check : 
        # print(study_id)
        count+=1
        flag = False     

print("Number of subjects without Axial T2 scan :", count)        

Number of subjects without Axial T2 scan : 0


In [12]:
# Check if every subject have Sagittal T2 volume

flag = True
count = 0
for i, study_id in enumerate(study_ids): 
    check=False
    for seq in sequence_lists[i]:
        if seq[0]=="Sagittal T2/STIR":
            check=True
            
    if not check : 
        # print(study_id)
        count+=1
        flag = False     

print("Number of subjects without Sagittal T2/STIR scan :", count)       

Number of subjects without Sagittal T2/STIR scan : 1


### Check if there are multiple volumes

The script below writes in a dictionary the number of nifti files per subject for each sequence type.

In [13]:
X = descriptions.groupby(["series_description"]).apply(lambda x: list(zip(x['study_id'], x['series_id']))).reset_index().values

In [14]:
count = {} 
for seq in sequence_types :
    count[seq] = []

for i, seq in enumerate(sequence_types) :
    for (id, serie_id) in tqdm(X[i, 1]) :
        c = len(os.listdir(os.path.join("data", "train_nifti", str(id), str(serie_id)))) // 2
        if c > 1 :
            count[seq].append((id, serie_id, c))

100%|██████████| 1974/1974 [00:00<00:00, 73014.53it/s]
